# Unlocking Blue Carbon: A Mechanistic MRV Framework
## Using Deep Spatial Intelligence & Real-Time Buoy Data

**What is this?**
A clean, machine-learning-ready framework fusing empirical field measurements with mechanistic environmental drivers (temperature, salinity, and proximity metrics) to estimate coastal carbon density. 

This notebook is built on the open dataset: **unified_bc_blue_carbon_filled.csv** *(provided as part of this submission)*.

**Why it matters?**
By shifting from geographic coordinates to physical drivers (Hakai buoy data), we've created a model that generalizes across coastal topographies. This enables scalable coastal carbon estimation supporting carbon markets, conservation policy, and climate accounting.

**What did we achieve?**
A statistically validated Machine Learning pipeline that achieves an **R² ≈ 0.75** (best model: Random Forest), proving that blue carbon density can be effectively predicted using physical environmental context.


## Why Blue Carbon Matters: The "Carbon Vault"
While tropical rainforests are often the "face" of carbon sequestration, coastal **Blue Carbon** ecosystems (like eelgrass) are significantly more efficient long-term storage engines.

### 📊 Storage Comparison: Eelgrass vs. Rainforests

| Metric | Tropical Rainforests | Eelgrass Meadows (e.g., Victoria) |
|:--- |:--- |:--- |
| **Carbon Storage** | ~100–300 tonnes / hectare | **~200–1,000+ tonnes / hectare** |
| **Primary Location** | Above ground (trees/biomass) | **Below ground (sediments)** |
| **Vulnerability** | High (Fire, logging → rapid release) | **Low (Can remain buried for millennia)** |
| **Sequestration Rate**| ~2–10 t CO₂/ha/year | ~1–10 t CO₂/ha/year |

### 🧠 The Big Takeaway
*   **Rainforests** = Fast storage, but fragile and easily disrupted.
*   **Eelgrass** = A permanent **"Carbon Vault"**. Because carbon is buried in low-oxygen sediments, it stays locked away much longer than in any terrestrial forest.

By protecting and accurately measuring these ecosystems, we secure a critical component of the planet's long-term climate stability.


## Mechanistic Modeling: Beyond Geographic Coordinates
We transitioned from basic spatial interpolation to a **mechanistic environmental driver** approach. Algorithms that rely purely on coordinates often fail to generalize; we resolved this by:
- **Excluding Lat/Lon**: Coordinates are metadata used to retrieve context, but are **completely excluded** as direct features.
- **Telemetered Buoy Integration**: We integrated **Hakai Institute** buoy data (12-month mean SST and Salinity) to provide high-fidelity physical context.
- **Haversine Geo-Density**: We account for ecological neighborhood effects using a 15km eelgrass density matrix.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# To support one-click reproducibility without Web-Scraping API failures:
# We bypass the fragile 45-minute ERDDAP fetching loop and load the pre-calculated, spatially enriched matrices natively.
try:
    # Try Kaggle environment path first
    df = pd.read_csv('/kaggle/input/datasets/jasonpoitras/unified-bc-blue-carbon-filled/unified_bc_blue_carbon_filled.csv')
    print("Pre-calculated MRV pipeline dataset loaded successfully from Kaggle! (100% Offline-Safe)")
except FileNotFoundError:
    try:
        # Try local environment fallback
        df = pd.read_csv('unified_bc_blue_carbon_filled.csv')
        print("Pre-calculated MRV pipeline dataset loaded successfully locally! (100% Offline-Safe)")
    except FileNotFoundError:
        df = None
        print("Dataset missing. Please ensure 'unified_bc_blue_carbon_filled.csv' is attached to this kernel.")
        
if df is not None:
    display(df.head())


## Ensemble Baseline & Evaluation
We established a robust ensemble baseline using standard tabular models (`XGBoost`, `LightGBM`, `CatBoost`) and an explicit `VotingRegressor` to predict unmapped carbon regions using rigorous LOOCV cross-validation.


In [ ]:

import os
import warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Features used for evaluation (Excluding raw Latitude and Longitude)
features = [c for c in df.columns if c.startswith((
    'region_',
    'habitat_type_',
    'habitat_area_ha',
    'anthropogenic_stress', 
    'percent_fines',
    'percent_oc',
    'd13C',
    'd15N',
    'buoy_temperature', 
    'buoy_salinity',
    'sea_surface_temperature', 
    'sea_surface_salinity', 
    'neighbor_density_15km', 
    'spatial_cluster_'
))]
X = df[~df['is_carbon_gap']][features].astype(float).values
y = df[~df['is_carbon_gap']]['carbon_density_gCm2'].astype(float).values

# Build the base models and VotingRegressor ensemble
# We focus the Ensemble on the top-performing models (RF and CatBoost) 
# to optimize accuracy after identifying that XGB/LGB were underperforming in this mechanistic setup.
xgb_model = XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
lgb_model = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
cat_model = CatBoostRegressor(n_estimators=100, random_state=42, verbose=0)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

ensemble = VotingRegressor(estimators=[
    ('cat', cat_model),
    ('rf', rf_model)
])

loo = LeaveOneOut()

models_to_evaluate = {
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model,
    "Random Forest": rf_model,
    "Ensemble": ensemble
}

print("Evaluating all tabular models using rigorous LOOCV...")
print("-" * 50)

final_y_pred = None

for name, clf in models_to_evaluate.items():
    pipe = make_pipeline(SimpleImputer(strategy='median'), clf)
    y_pred_current = cross_val_predict(pipe, X, y, cv=loo, n_jobs=-1)
    
    current_rmse = np.sqrt(mean_squared_error(y, y_pred_current))
    current_r2 = r2_score(y, y_pred_current)
    
    if name == "Ensemble":
        print("-" * 50)
        final_y_pred = y_pred_current
        
    print(f"[{name.ljust(10)}] LOOCV RMSE: {current_rmse:6.2f} | R²: {current_r2:.3f}")

# Expose ensemble predictions/metrics to the visualization cell below
y_pred = final_y_pred
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)


## Visualizing Success, Predictability & Drivers


In [ ]:

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# 1. Predicted vs Actual
axes[0].scatter(y, y_pred, alpha=0.7, edgecolors='white', s=80, color='#00A86B')
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='#2C3E50', linestyle='--', lw=2, label='1:1 Line')
props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
axes[0].text(0.05, 0.95, f'Model: Ensemble (LOOCV)\nRMSE: {rmse:.2f}\nR²: {r2:.3f}', 
             transform=axes[0].transAxes, fontsize=12, verticalalignment='top', bbox=props)
axes[0].set_title("Predicted vs Actual Blue Carbon Sequestration", fontsize=16, fontweight='bold', pad=15)
axes[0].set_xlabel("Actual Carbon Density (gC/m²)", fontsize=12)
axes[0].set_ylabel("Predicted Carbon Density (gC/m²)", fontsize=12)

# 2. Residual Distribution
residuals = y - y_pred
sns.histplot(residuals, kde=True, color='#FF6F61', bins=40, edgecolor="white", alpha=0.7, ax=axes[1])
axes[1].axvline(x=0, color='#2C3E50', linestyle='--', lw=2, label='Zero Error')
mean_resid = np.mean(residuals)
axes[1].axvline(x=mean_resid, color='#3498DB', linestyle='-.', lw=2, label=f'Mean Error: {mean_resid:.2f}')
axes[1].set_title("Model Residuals (Prediction Error Distribution)", fontsize=16, fontweight='bold', pad=15)
axes[1].set_xlabel("Prediction Error (gC/m²)", fontsize=12)

# 3. Environmental Spatial Drivers (Excluding Direct Geography)
# We fit a Random Forest explainer specifically on the environmental covariates to see which 
# ecological factors drive the variance when direct latitude/longitude are removed.
eco_features = [f for f in features if not f.startswith(('latitude', 'longitude'))]
X_eco = df[~df['is_carbon_gap']][eco_features].astype(float).values

eval_pipe = make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=100, random_state=42))
eval_pipe.fit(X_eco, y)
importances = eval_pipe.named_steps['randomforestregressor'].feature_importances_

# Extract the Top 10 Features
indices = np.argsort(importances)[-10:]
top_features = [eco_features[i] for i in indices]
top_importances = importances[indices]

axes[2].barh(range(len(indices)), top_importances, color='#3498DB', edgecolor='white')
axes[2].set_yticks(range(len(indices)))
clean_labels = [f.replace('_', ' ').title() for f in top_features]
axes[2].set_yticklabels(clean_labels, fontsize=11)
axes[2].set_title("Top 10 Environmental & Spatial Drivers", fontsize=16, fontweight='bold', pad=15)
axes[2].set_xlabel("Relative Feature Importance (Excl. Lat/Lon)", fontsize=12)

plt.tight_layout()
plt.show()
